In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install transformers peft trl datasets bitsandbytes accelerate
!pip install --no-deps trl peft accelerate bitsandbytes

In [4]:
import torch
from tqdm import tqdm
import sys
import glob
import os
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

In [5]:
JSON_SCHEMA = '/content/drive/MyDrive/Rohan/DataLens/BIRD_SQL/minidev/MINIDEV/dev_tables.json'
JSON_QA_PAIRS = '/content/drive/MyDrive/Rohan/DataLens/BIRD_SQL/minidev/MINIDEV/mini_dev_*_with_schema.json'
DB_PARENT_PATH = '/content/drive/MyDrive/Rohan/DataLens/BIRD_SQL/minidev/MINIDEV/dev_databases/'

In [6]:
model_id = "Qwen/Qwen2.5-Coder-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [7]:
json_files = glob.glob(JSON_QA_PAIRS)
print(f"Found the following JSON datasets: {json_files}")

def format_prompt_completion(example):
    db_id = example.get('db_id', '')
    question = example.get('question', '')
    evidence = example.get('evidence', '')
    schema = example.get('schema', '')
    sql_query = example.get('SQL', example.get('query', ''))
    db_path = os.path.join(DB_PARENT_PATH, db_id, db_id+'.sqlite')

    prompt_text = f"Write a SQL query to answer the following question.\n\nEvidence: {evidence}\nSchema: {schema} \nQuestion: {question}"

    # 1. Structure just the system and user messages
    prompt_messages = [
        {"role": "system", "content": "You are an expert SQL coding assistant. Given a natural langauge question along with the database schema and evidence, generate an SQL query. PLEASE JUST GENERATE THE SQL QUERY AND NOTHING ELSE."},
        {"role": "user", "content": prompt_text}
    ]

    # 2. Convert the prompt into a string using the tokenizer
    # (add_generation_prompt=True automatically appends the <|im_start|>assistant\n tag)
    prompt_str = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)

    # 3. Define the completion string (SQL + the End of Sequence token so the model learns when to stop)
    completion_str = sql_query + tokenizer.eos_token

    # Return exactly the keys that modern SFTTrainer expects
    return {"prompt": prompt_str, "completion": completion_str, "db_id": db_id, "gt_sql": sql_query}

dataset = load_dataset("json", data_files=json_files, split="train")
print(f"Loaded a total of {len(dataset)} items across all files.")

formatted_dataset = dataset.map(format_prompt_completion)


Found the following JSON datasets: ['/content/drive/MyDrive/Rohan/DataLens/BIRD_SQL/minidev/MINIDEV/mini_dev_sqlite_with_schema.json', '/content/drive/MyDrive/Rohan/DataLens/BIRD_SQL/minidev/MINIDEV/mini_dev_postgresql_with_schema.json', '/content/drive/MyDrive/Rohan/DataLens/BIRD_SQL/minidev/MINIDEV/mini_dev_mysql_with_schema.json']


Generating train split: 0 examples [00:00, ? examples/s]

Loaded a total of 1500 items across all files.


Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [8]:
print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",           # Automatically places model on GPU if available
    torch_dtype=torch.bfloat16,  # Reduces memory footprint
)
model.eval()
print(f"Model loaded successfully on {model.device}!")

Loading model...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully on cuda:0!


In [9]:
import re
import sqlite3

def compare_results(gt_results, gen_results):
    if gt_results is None or gen_results is None:
        return False
    try:
        return set(gt_results) == set(gen_results)
    except TypeError:
        return gt_results == gen_results

import threading
import os
import sqlite3

def run_query_on_db(db_path, query, timeout_sec=10.0):
    """Executes a SQL query on a SQLite DB with a timeout to prevent hangs."""
    try:
        # Check if DB exists before trying to connect
        if not os.path.exists(db_path):
            return None, f"Database file not found at: {db_path}"

        # Connect to the database in Read-Only mode
        uri = f"file:{db_path}?mode=ro"
        conn = sqlite3.connect(uri, uri=True)
        cursor = conn.cursor()

        # 1. Set up a timer to interrupt the connection if it exceeds timeout_sec
        timer = threading.Timer(timeout_sec, conn.interrupt)
        timer.start()

        try:
            # 2. Execute the query
            cursor.execute(query)
            results = cursor.fetchall()
            error = None
        except sqlite3.OperationalError as e:
            # 3. Catch the interrupt and log it as a timeout error
            if "interrupted" in str(e).lower():
                results = None
                error = f"Execution timed out after {timeout_sec} seconds."
            else:
                results = None
                error = str(e)
        except Exception as e:
            results = None
            error = str(e)
        finally:
            # 4. CRITICAL: Cancel the timer if the query finishes fast enough!
            timer.cancel()
            conn.close()

        return results, error

    except Exception as e:
        return None, str(e)

def clean_generated_sql(sql_str):
    """Removes markdown code blocks to ensure SQLite can run the raw string."""
    sql_str = re.sub(r"```sql\s*", "", sql_str, flags=re.IGNORECASE)
    sql_str = re.sub(r"```\s*", "", sql_str)
    return sql_str.strip()

def generate_sql(prompt_text):
    # Tokenize input and move to the same device as the model
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.0,       # Greedy decoding (0.0) is best for exact SQL syntax
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the newly generated tokens (ignoring the prompt)
    input_length = inputs['input_ids'].shape[1]
    generated_tokens = outputs[0][input_length:]
    raw_output = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return clean_generated_sql(raw_output)


In [10]:
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

BATCH_SIZE = 8
total_queries = len(formatted_dataset)
correct_count = 0
error_count = 0

print(f"Starting Batched Evaluation for {total_queries} queries (Batch Size: {BATCH_SIZE})...\n")

for i in tqdm(range(0, total_queries, BATCH_SIZE), desc="Processing Batches"):
    # Slice the dataset for the current batch
    batch_rows = [formatted_dataset[j] for j in range(i, min(i + BATCH_SIZE, total_queries))]
    batch_prompts = [row['prompt'] for row in batch_rows]

    # --- STEP 1: Batched LLM Generation ---
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True).to(model.device)
    input_lengths = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    for idx, row in enumerate(batch_rows):
        # Extract only the newly generated tokens for this specific item in the batch
        generated_tokens = outputs[idx][input_lengths:]
        raw_output = tokenizer.decode(generated_tokens, skip_special_tokens=True)
        generated_sql = clean_generated_sql(raw_output)

        # Get ground truth and DB metadata
        gt_sql = row.get('gt_sql', row.get('SQL', ''))
        current_db_id = row['db_id']
        db_path = os.path.join(DB_PARENT_PATH, current_db_id, f"{current_db_id}.sqlite")

        #print(f"\nGenerated SQL: {generated_sql}, GT Query: {gt_sql}, db_id: {current_db_id}\n")
        # Execute queries against SQLite
        gt_results, gt_error = run_query_on_db(db_path, gt_sql)
        model_results, model_error = run_query_on_db(db_path, generated_sql)
        #print(f"\nModel Result: {model_results, model_error}, GT_Result: {gt_results, gt_error}\n")

        # Evaluate performance
        is_correct = False
        if not gt_error and not model_error:
            is_correct = compare_results(gt_results, model_results)
        elif model_error:
            error_count += 1

        if is_correct:
            correct_count += 1

# 3. Print Final Accuracy
accuracy = (correct_count / total_queries) * 100 if total_queries > 0 else 0

print("\n" + "="*40)
print("🏆 FINAL EVALUATION METRICS 🏆")
print("="*40)
print(f"Total Evaluated: {total_queries}")
print(f"Exact Execution Matches: {correct_count}")
print(f"Model Execution Errors: {error_count}")
print(f"Execution Accuracy (EX): {accuracy:.2f}%")
print("="*40)

Starting Batched Evaluation for 1500 queries (Batch Size: 8)...



Processing Batches: 100%|██████████| 188/188 [24:50<00:00,  7.93s/it]


🏆 FINAL EVALUATION METRICS 🏆
Total Evaluated: 1500
Exact Execution Matches: 636
Model Execution Errors: 240
Execution Accuracy (EX): 42.40%
